# C-BARR: Colab & Kaggle End-to-End GPU Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dolong2110/C-BARR/blob/master/notebooks/C_BARR_Colab_Kaggle.ipynb)

This notebook clones `dolong2110/C-BARR`, verifies/downloads datasets and dependencies, and executes the complete **C-BARR** pipeline on **Google Colab** (`T4 16GB` / `L4 24GB` / `A100 40GB`) or **Kaggle** (`P100 16GB` / `T4 16GB`):

- **Hardware-Aware Precision (`DTYPE = "auto"`)**: Uses native `bfloat16` on Ampere+ GPUs (`A100`, `L4`, `RTX 4090`) and switches automatically to `float16` on `T4` / `P100`.
- **VRAM Protection**: `Qwen/Qwen3.5-9B` (~18.5 GB) is auto-enabled only when `VRAM >= 22 GB` so 16 GB Colab/Kaggle sessions do not OOM.
- **Crash-Safe Resume**: Every completed item is flushed per UID into `OUT_DIR/<model>/*_cache.jsonl`. On Colab, mounting Google Drive preserves all progress across disconnects; on Kaggle, HF model caches go into temporary disk (`/tmp/hf_cache` or `/kaggle/temp`) so `/kaggle/working` stays within quota.

In [ ]:
import os
import pathlib
import platform
import shutil
import subprocess
import sys

# Detect runtime platform (Google Colab, Kaggle, or standalone Linux)
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or pathlib.Path("/kaggle/working").exists()
PLATFORM = "colab" if IS_COLAB else ("kaggle" if IS_KAGGLE else "linux")

# Toggle whether Colab mounts Google Drive so intermediate caches survive disconnects
USE_GOOGLE_DRIVE = True
DRIVE_ROOT = None

if IS_COLAB:
    WORKSPACE_ROOT = pathlib.Path("/content")
    if USE_GOOGLE_DRIVE:
        try:
            from google.colab import drive

            drive.mount("/content/drive", force_remount=False)
            DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/C-BARR")
            DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        except Exception as exc:
            print(f"[WARN] Google Drive mount skipped ({exc}); using ephemeral /content storage.")
elif IS_KAGGLE:
    WORKSPACE_ROOT = pathlib.Path("/kaggle/working")
else:
    WORKSPACE_ROOT = pathlib.Path.cwd()

REPO_DIR = WORKSPACE_ROOT / "C-BARR"
OUT_DIR = (DRIVE_ROOT / "outputs") if DRIVE_ROOT else (REPO_DIR / "outputs")
FIG_DIR = REPO_DIR / "paper" / "figures"

# Store large HF weight downloads outside /kaggle/working so output quota is preserved for caches & tables
if IS_KAGGLE:
    hf_tmp = pathlib.Path("/kaggle/temp/hf_cache")
    hf_tmp.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(hf_tmp)
elif IS_COLAB:
    os.environ.setdefault("HF_HOME", "/tmp/hf_cache")

# Avoid matplotlib font-cache permission warnings on ephemeral containers
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
pathlib.Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import torch

HAS_CUDA = torch.cuda.is_available()
VRAM_GB = 0.0
GPU_NAME = "None (CPU)"
SM_CAP = (0, 0)
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    VRAM_GB = props.total_memory / (1024**3)
    SM_CAP = torch.cuda.get_device_capability(0)

disk = shutil.disk_usage(str(WORKSPACE_ROOT))
print(f"Platform       : {PLATFORM} ({platform.machine()})")
print(f"Python / Torch : {sys.version.split()[0]} / {torch.__version__}")
print(f"GPU            : {GPU_NAME} | VRAM={VRAM_GB:.1f} GB | SM={SM_CAP} | BF16={HAS_CUDA and torch.cuda.is_bf16_supported() and SM_CAP[0] >= 8}")
print(f"Workspace / Out: {REPO_DIR} -> {OUT_DIR}")
print(f"HF_HOME        : {os.environ.get('HF_HOME', 'default (~/.cache/huggingface)')}")
print(f"Free Disk      : {disk.free / (1024**3):.1f} GB")
if not HAS_CUDA:
    print("[WARN] No CUDA GPU detected! Turn on GPU Accelerator in Runtime / Session settings before Phase 4-6.")

In [ ]:
# === Experiment & Pipeline Configuration ===
os.environ.setdefault("CBARR_COLOR", "1")  # Highlight key metrics/gates in Colab/Kaggle cells
REPO_URL = "https://github.com/dolong2110/C-BARR.git"
REPO_BRANCH = "master"

# Primary evaluation + causal diagnostic models (strict341 -> diag491)
MODELS_PRIMARY = [
    "Qwen/Qwen3.5-4B",
    "HuggingFaceTB/SmolLM3-3B",
]

# Cross-scale models (run with --strict-diag for Table 2)
MODELS_SCALE = [
    "Qwen/Qwen3.5-0.8B",
    "Qwen/Qwen3.5-2B",
]
# Automatically include Qwen3.5-9B (~18.5 GB) when GPU has >= 22 GB VRAM (L4/A100)
AUTO_INCLUDE_9B = True
FORCE_9B = False
if (AUTO_INCLUDE_9B and VRAM_GB >= 22.0) or FORCE_9B:
    MODELS_SCALE.append("Qwen/Qwen3.5-9B")

# Pipeline stage toggles
RUN_PREFLIGHT_TESTS = True
RUN_LIVE_SMOKE = HAS_CUDA
RUN_CALIBRATE_ONLY = True
RUN_PRIMARY_MATRIX = True
RUN_SCALE_SWEEP = True
INCLUDE_B512 = False       # Contingency R3 auxiliary B=512 sweep
RUN_W7_EXTENSIONS = False  # Optional W7: DeltaNet state blend + Prefill MLP probe

# Hardware & cache behavior
DTYPE = "auto"                        # "auto" picks BF16 on L4/A100 and FP16 on T4/P100
INSTALL_FAST_DELTANET_KERNELS = "auto"  # "auto" installs + GPU-verifies Triton/wheel DeltaNet kernels on sm >= 7.5
AUTO_PRUNE_HF_CACHE_BELOW_GB = 18.0   # Prune old downloaded weights only when disk drops below threshold
PREFETCH_MODEL_WEIGHTS = False        # Set True to pre-download all HF checkpoints in Cell 5


def _load_secret(name: str) -> str:
    val = os.environ.get(name, "").strip()
    if val:
        return val
    if IS_COLAB:
        try:
            from google.colab import userdata

            s = userdata.get(name)
            if s:
                return str(s).strip()
        except Exception:
            pass
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient

            s = UserSecretsClient().get_secret(name)
            if s:
                return str(s).strip()
        except Exception:
            pass
    return ""


GITHUB_TOKEN = _load_secret("GITHUB_TOKEN")  # Needed only if repository is private
HF_TOKEN = _load_secret("HF_TOKEN")          # Optional; improves HF Hub download limits
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login

        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face authentication configured cleanly.")
    except Exception as exc:
        print(f"[WARN] huggingface_hub login skipped: {exc}")

ALL_ACTIVE_MODELS = list(dict.fromkeys(MODELS_PRIMARY + (MODELS_SCALE if RUN_SCALE_SWEEP else [])))
print("Active models:", ALL_ACTIVE_MODELS)

In [ ]:
import importlib.metadata
import re


def _sanitize_git_err(msg: str, token: str) -> str:
    return msg.replace(token, "***") if token else msg


def _git_clone_url() -> str:
    if GITHUB_TOKEN and REPO_URL.startswith("https://"):
        return REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1)
    return REPO_URL


# 1. Clone or pull latest C-BARR repository
if not (REPO_DIR / ".git").exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    proc = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", _git_clone_url(), str(REPO_DIR)],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(_sanitize_git_err(proc.stderr or proc.stdout, GITHUB_TOKEN))
    print(f"Cloned {REPO_URL} -> {REPO_DIR}")
else:
    # Restore tracked smoke/report artifacts modified by prior runs before fast-forwarding
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--", "outputs"], check=False)
    proc = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(_sanitize_git_err(proc.stderr or proc.stdout, GITHUB_TOKEN))
    print(f"Updated existing repo at {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


# 2. Install/upgrade only missing runtime packages (preserve container CUDA PyTorch!)
def _pkg_ver(name: str) -> tuple[int, ...] | None:
    try:
        v = importlib.metadata.version(name)
        nums = [int(x) for x in re.findall(r"\d+", v)[:2]]
        return tuple(nums)
    except importlib.metadata.PackageNotFoundError:
        return None


required_pkgs = [
    "transformers>=4.53.0",
    "accelerate",
    "datasets",
    "matplotlib",
    "scipy",
    "scikit-learn",
    "nltk",
    "immutabledict",
    "langdetect",
    "absl-py",
]
torch_v = _pkg_ver("torch")
if torch_v is None or torch_v < (2, 6):
    required_pkgs.insert(0, "torch>=2.6")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *required_pkgs], check=True)

# Optional fast CUDA/Triton Gated DeltaNet kernels with live CUDA self-test & safe rollback
kernel_mode = (
    "always"
    if INSTALL_FAST_DELTANET_KERNELS is True
    else ("off" if INSTALL_FAST_DELTANET_KERNELS is False else str(INSTALL_FAST_DELTANET_KERNELS))
)
if HAS_CUDA and kernel_mode != "off":
    subprocess.run(
        [sys.executable, "scripts/setup_fast_kernels.py", "--mode", kernel_mode],
        check=False,
    )

import importlib.util as _ilu

has_fla = _ilu.find_spec("fla") is not None
has_cc1d = _ilu.find_spec("causal_conv1d") is not None
print(f"Environment provisioning complete (fla={has_fla}, causal_conv1d={has_cc1d}).")

In [ ]:
from langdetect import DetectorFactory
import nltk
from transformers import AutoModelForCausalLM, AutoTokenizer

# Patch D3: Pin langdetect seed globally
DetectorFactory.seed = 0

# 1. Verify official google-research instruction_following_eval is present (clone sparse if missing)
ifeval_dir = REPO_DIR / "instruction_following_eval"
if not ifeval_dir.exists():
    tmp_gr = pathlib.Path("/tmp/gr_sparse")
    if tmp_gr.exists():
        shutil.rmtree(tmp_gr)
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--filter=blob:none",
            "--sparse",
            "https://github.com/google-research/google-research.git",
            str(tmp_gr),
        ],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(tmp_gr), "sparse-checkout", "set", "instruction_following_eval"],
        check=True,
    )
    shutil.copytree(tmp_gr / "instruction_following_eval", ifeval_dir)
    shutil.rmtree(tmp_gr)

# 2. Download NLTK packages required by IFEval verifiers
for nltk_res in ("punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger_eng"):
    nltk.download(nltk_res, quiet=True)

# 3. Verify frozen dataset splits against manifest_sha256.json (never re-randomize frozen splits!)
from scripts.run_matrix import REQUIRED_DATA_SPLITS, verify_manifest_integrity

data_dir = REPO_DIR / "data"
if not (data_dir / "manifest_sha256.json").exists() or not all(
    (data_dir / f).exists() for f in REQUIRED_DATA_SPLITS
):
    print("Frozen splits missing; building deterministically via data/freeze_datasets.py (seed=42)...")
    subprocess.run([sys.executable, str(data_dir / "freeze_datasets.py")], check=True)

manifest = verify_manifest_integrity(data_dir)
print(f"Verified {len(manifest)} frozen dataset files via SHA-256 manifest:")
for fname in REQUIRED_DATA_SPLITS:
    print(f"  OK {fname}: {manifest[fname][:16]}...")

# 4. Pre-download Hugging Face tokenizers (and optionally full checkpoints)
for m_id in ALL_ACTIVE_MODELS:
    print(f"Checking tokenizer cache for {m_id}...")
    AutoTokenizer.from_pretrained(m_id)
    if PREFETCH_MODEL_WEIGHTS:
        print(f"Pre-downloading weights for {m_id}...")
        AutoModelForCausalLM.from_pretrained(m_id)

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import json

if RUN_PREFLIGHT_TESTS:
    print("=== [Gate 0] Running Offline Preflight Unit Tests ===")
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

if RUN_LIVE_SMOKE:
    print("=== [Step 0a] Running Live GPU 12-Item Smoke Gate ===")
    smoke_models = [m for m in MODELS_PRIMARY if ("9B" not in m or VRAM_GB >= 22.0)] or [ALL_ACTIVE_MODELS[0]]
    subprocess.run(
        [sys.executable, "scripts/run_smoke_test.py", "--live-gpu", "--models", *smoke_models],
        check=True,
    )

In [ ]:
if RUN_CALIBRATE_ONLY:
    cal_model = MODELS_PRIMARY[0] if MODELS_PRIMARY else ALL_ACTIVE_MODELS[0]
    print(f"=== [Step 0b] Running Zero-Leakage Calibration on {cal_model} ===")
    subprocess.run(
        [
            sys.executable,
            "scripts/run_matrix.py",
            "--model",
            cal_model,
            "--data-dir",
            "data",
            "--out-dir",
            str(OUT_DIR),
            "--dtype",
            DTYPE,
            "--calibrate-only",
        ],
        check=True,
    )

In [ ]:
def _prune_hf_hub_if_low_disk() -> None:
    hf_hub = pathlib.Path(os.environ.get("HF_HOME", pathlib.Path.home() / ".cache" / "huggingface")) / "hub"
    if not hf_hub.exists():
        return
    free_gb = shutil.disk_usage(str(hf_hub)).free / (1024**3)
    if free_gb < AUTO_PRUNE_HF_CACHE_BELOW_GB:
        print(f"[Disk Guard] Free disk {free_gb:.1f} GB < {AUTO_PRUNE_HF_CACHE_BELOW_GB:.1f} GB; pruning HF weight cache...")
        for item in hf_hub.iterdir():
            if item.is_dir() and item.name.startswith("models--"):
                shutil.rmtree(item, ignore_errors=True)


def run_model_matrix(model_id: str, strict_diag: bool, include_b512: bool = False) -> None:
    cmd = [
        sys.executable,
        "scripts/run_matrix.py",
        "--model",
        model_id,
        "--data-dir",
        "data",
        "--out-dir",
        str(OUT_DIR),
        "--dtype",
        DTYPE,
    ]
    if strict_diag:
        cmd.append("--strict-diag")
    if include_b512:
        cmd.append("--include-b512")
    print("Executing:", " ".join(cmd))
    subprocess.run(cmd, check=True)


if RUN_PRIMARY_MATRIX:
    for idx, model_id in enumerate(MODELS_PRIMARY):
        if idx > 0:
            _prune_hf_hub_if_low_disk()
        print(f"\n=== [Phase 4/5] Primary Matrix: {model_id} (strict341 -> diag491) ===")
        # Running strict341 first populates 70% of diag491 controls cache automatically
        run_model_matrix(model_id, strict_diag=True, include_b512=INCLUDE_B512)
        run_model_matrix(model_id, strict_diag=False, include_b512=INCLUDE_B512)

In [ ]:
if RUN_SCALE_SWEEP:
    for model_id in MODELS_SCALE:
        _prune_hf_hub_if_low_disk()
        print(f"\n=== [Phase 6] Scale Sweep: {model_id} (strict341) ===")
        run_model_matrix(model_id, strict_diag=True, include_b512=INCLUDE_B512)

if RUN_W7_EXTENSIONS:
    w7_model = MODELS_PRIMARY[0] if MODELS_PRIMARY else ALL_ACTIVE_MODELS[0]
    print(f"\n=== [Optional W7] Running DeltaNet Blend & Prefill MLP on {w7_model} ===")
    subprocess.run(
        [sys.executable, "scripts/ext_deltanet_blend.py", "--model", w7_model, "--out-dir", str(OUT_DIR)],
        check=True,
    )
    subprocess.run(
        [sys.executable, "scripts/ext_prefill_mlp.py", "--model", w7_model, "--out-dir", str(OUT_DIR)],
        check=True,
    )

In [ ]:
import zipfile

print("=== [Phase 7] Statistical Inference, LaTeX Tables & Vector Figures ===")
subprocess.run(
    [
        sys.executable,
        "scripts/analyze_and_plot.py",
        "--out-dir",
        str(OUT_DIR),
        "--fig-dir",
        str(FIG_DIR),
    ],
    check=True,
)

print("=== [Audit] Double-Blind Anonymity Check ===")
subprocess.run(["bash", "scripts/check_anonymity.sh"], check=True)

# Print quick Gate-4 summary across evaluated models
stats_path = OUT_DIR / "stats_summary.json"
stats = json.loads(stats_path.read_text(encoding="utf-8"))
for m_name, m_data in stats.get("models", {}).items():
    g4 = m_data["gate_audit"]["gate4_primary"]
    boot = m_data.get("paired_bootstrap_95ci", {}).get("M2_minus_B1b", {})
    oaa_d = boot.get("oaa_250", {})
    print(
        f"[{m_name}] M2>=B1b Macro={g4['m2_macro_ge_b1b']} | "
        f"TokRed={g4['token_reduction_pct']:.1f}% | "
        f"p_holm={g4['m2_p_holm_vs_b1b']:.4g} | "
        f"Delta OAA_250={100*oaa_d.get('diff_b_minus_a', 0):.1f} pp "
        f"[{100*oaa_d.get('ci95_lo', 0):.1f}, {100*oaa_d.get('ci95_hi', 0):.1f}]"
    )

# Package compact downloadable bundle (tables, stats JSONs, per-model summaries, 3 PDF figures)
bundle_zip = WORKSPACE_ROOT / "cbarr_results_bundle.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for top_file in ("stats_summary.json", "paper_tables.tex"):
        p = OUT_DIR / top_file
        if p.exists():
            zf.write(p, arcname=f"outputs/{top_file}")
    for sub in OUT_DIR.iterdir():
        if sub.is_dir():
            for fname in (
                "calibration_summary.json",
                "matrix_summary.json",
                "matrix_summary_strict341.json",
                "matrix_summary_diag491.json",
                "tables_rows.tex",
                "tables_rows_strict341.tex",
                "tables_rows_diag491.tex",
                "ext_deltanet_blend.json",
                "prefill_mlp_summary.json",
            ):
                fp = sub / fname
                if fp.exists():
                    zf.write(fp, arcname=f"outputs/{sub.name}/{fname}")
    for pdf in sorted(FIG_DIR.glob("*.pdf")):
        zf.write(pdf, arcname=f"paper/figures/{pdf.name}")

print(f"Packaged results bundle: {bundle_zip} ({bundle_zip.stat().st_size / 1024:.1f} KB)")
if IS_COLAB:
    try:
        from google.colab import files

        files.download(str(bundle_zip))
    except Exception as exc:
        print(f"[INFO] Download file manually from Colab Files pane: {bundle_zip} ({exc})")